<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/05_applications/resume_matching_score_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import re

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
priority_skills = [
    "machine learning",
    "deep learning",
    "nlp",
    "python",
    "data science"
]

In [ ]:
job_description = """
We are hiring a machine learning engineer with strong experience in
Python, NLP, deep learning, and building intelligent systems.
"""

In [ ]:
resumes = [
    """
    Data scientist with experience in machine learning, NLP, and Python.
    Built semantic search systems and worked on deep learning models.
    """,

    """
    Frontend developer skilled in HTML, CSS, JavaScript, and React.
    Focused on UI development and web performance.
    """,

    """
    Machine learning engineer with expertise in Python, deep learning,
    NLP pipelines, and AI-based recommendation systems.
    """,

    """
    Software engineer experienced in Java, SQL, and backend systems.
    Worked on database optimization and APIs.
    """
]

In [ ]:
def extract_skills(text, skill_list):
    text = text.lower()
    return [
        skill for skill in skill_list
        if re.search(rf"\b{skill}\b", text)
    ]

In [ ]:
job_skills = extract_skills(job_description, priority_skills)

resume_skills = [
    extract_skills(resume, priority_skills)
    for resume in resumes
]

job_skills, resume_skills

In [ ]:
job_embedding = model.encode([job_description])
resume_embeddings = model.encode(resumes)

In [16]:
semantic_scores = cosine_similarity(job_embedding, resume_embeddings)[0]
semantic_scores

array([0.6973975 , 0.34983933, 0.7581085 , 0.51140726], dtype=float32)

In [17]:
skill_scores = []

for skills in resume_skills:
    if len(job_skills) == 0:
        skill_scores.append(0)
    else:
        skill_scores.append(len(set(skills) & set(job_skills)) / len(job_skills))

skill_scores

[1.0, 0.0, 1.0, 0.0]

In [18]:
final_scores = [
    (0.7 * sem) + (0.3 * skill)
    for sem, skill in zip(semantic_scores, skill_scores)
]

final_scores

[np.float32(0.7881782),
 np.float32(0.24488753),
 np.float32(0.83067596),
 np.float32(0.35798508)]

In [19]:
def interpret_score(score):
    if score >= 0.75:
        return "Strong Match ✅"
    elif score >= 0.5:
        return "Moderate Match ⚠️"
    else:
        return "Weak Match ❌"

In [20]:
results = pd.DataFrame({
    "Resume Index": range(len(resumes)),
    "Semantic Score": semantic_scores,
    "Skill Match Score": skill_scores,
    "Final Score": final_scores,
    "Match Interpretation": [interpret_score(s) for s in final_scores]
})

results = results.sort_values(by="Final Score", ascending=False)
results

,Resume Index,Semantic Score,Skill Match Score,Final Score,Match Interpretation
2,2,0.758108,1.0,0.830676,Strong Match ✅
0,0,0.697397,1.0,0.788178,Strong Match ✅
3,3,0.511407,0.0,0.357985,Weak Match ❌
1,1,0.349839,0.0,0.244888,Weak Match ❌


In [21]:
for _, row in results.iterrows():
    i = int(row["Resume Index"])
    print("Resume Index:", i)
    print("Semantic Score:", round(row["Semantic Score"], 3))
    print("Skill Match Score:", round(row["Skill Match Score"], 3))
    print("Final Score:", round(row["Final Score"], 3))
    print("Match Type:", row["Match Interpretation"])
    print("Matched Skills:", resume_skills[i])
    print("Resume Text:", resumes[i])
    print("-" * 80)

Resume Index: 2
Semantic Score: 0.758
Skill Match Score: 1.0
Final Score: 0.831
Match Type: Strong Match ✅
Matched Skills: ['machine learning', 'deep learning', 'nlp', 'python']
Resume Text: 
    Machine learning engineer with expertise in Python, deep learning,
    NLP pipelines, and AI-based recommendation systems.
    
--------------------------------------------------------------------------------
Resume Index: 0
Semantic Score: 0.697
Skill Match Score: 1.0
Final Score: 0.788
Match Type: Strong Match ✅
Matched Skills: ['machine learning', 'deep learning', 'nlp', 'python']
Resume Text: 
    Data scientist with experience in machine learning, NLP, and Python.
    Built semantic search systems and worked on deep learning models.
    
--------------------------------------------------------------------------------
Resume Index: 3
Semantic Score: 0.511
Skill Match Score: 0.0
Final Score: 0.358
Match Type: Weak Match ❌
Matched Skills: []
Resume Text: 
    Software engineer experienced in

- This notebook analyzes resume-job matching results by breaking down semantic similarity, skill-based matching, and final weighted scores.
- It adds explainability to the system by showing why a resume was ranked high or low, making the model outputs more transparent and interpretable for real-world use.
